In [11]:
import numpy as np

In [ ]:
class Net:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")

        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # linear / identity
        return A

    @staticmethod
    def _df2(A):  # d(linear) == 1
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = self.X @ self.W1  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = self.Y1 @ self.W2  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # 1/(2N) * Sum (y-y_true)^2, mean over batch
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1

        self.W2 += -lr * self.dL_dW2
        self.W1 += -lr * self.dL_dW1


nn = Net(x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
W2: float64 (16, 2)


In [53]:
# Fixed-memory version of `Net`: ALL arrays are allocated ONCE in __init__, and the
# forward/backward hot path only writes into those preallocated buffers in place
# (no per-pass allocation). This mirrors how you'd write it in plain C: malloc every
# matrix up front, then each pass is just BLAS gemm calls with a preallocated output
# plus elementwise loops.
#
# Consequences / how it differs from Net:
#   - The batch size is FIXED at construction (buffers are sized (b_sz, ...)). Inputs
#     must be exactly (b_sz, x_sz); `forward` copies them into the preallocated X buffer.
#   - The big matmuls write into preallocated buffers via np.dot(A, B, out=C),
#     i.e. C = A @ B (a BLAS gemm into fixed memory, the C analogue).
#   - Activations stay GENERIC and swappable: _f1/_df1/_f2/_df2 are the same plain
#     functions as in Net. To keep them simple and easy to replace, they return a
#     fresh array and we copy it into the buffer (self.Y1[:] = self._f1(self.Z1)).
#     That copy allocates a small temp -- intentionally NOT optimized here. Making the
#     activations fully in-place / allocation-free (e.g. np.maximum(Z,0,out=Y) for ReLU,
#     np.sign(Y1,out=mask) for its derivative) is left as an exercise for the reader.
#   - The SGD step scales the gradient buffers in place by lr (a "fused" update), so
#     after backward() dL_dW* hold lr*grad rather than the raw grad -- fine for training,
#     and it avoids allocating a `lr * grad` temporary.
#   - `_loss` is a diagnostic and also allocates a tiny temp.
class NetFixedMem:
    def __init__(self, *, b_sz, x_sz, y1_sz, y2_sz):
        # --- parameters (weights) ---
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

        # --- preallocated forward buffers ---
        # X ~ (b_sz, x_sz)
        self.X = np.zeros((b_sz, x_sz))  # input copied here each forward
        self.Z1 = np.zeros((b_sz, y1_sz))  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = np.zeros((b_sz, y1_sz))  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = np.zeros((b_sz, y2_sz))  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = np.zeros((b_sz, y2_sz))  # Y2 ~ (b_sz, y2_sz)

        # --- preallocated backward buffers ---
        # E2 ~ (b_sz, y2_sz)
        self.E2 = np.zeros((b_sz, y2_sz))  # dL/dZ2
        # E1 ~ (b_sz, y1_sz)
        self.E1 = np.zeros((b_sz, y1_sz))  # dL/dZ1
        self.dL_dW2 = np.zeros((y1_sz, y2_sz))  # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW1 = np.zeros((x_sz, y1_sz))  # dL_dW1 ~ (x_sz, y1_sz)
        for name in ("X", "Z1", "Y1", "Z2", "Y2", "E2", "E1", "dL_dW1", "dL_dW2"):
            a = getattr(self, name)
            print(f"{name}: {a.dtype} {a.shape}")

    # --- generic, swappable activations (kept simple on purpose; replace at will) ---
    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # linear / identity
        return A

    @staticmethod
    def _df2(A):  # d(linear) == 1
        return np.ones_like(A)

    @staticmethod
    def _loss(Y_predicted, Y_true):  # diagnostic only (allocates a small temp)
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def forward(self, X):
        # X ~ (b_sz, x_sz)
        np.copyto(self.X, X)  # X buffer <- input (fixed batch)
        # Z1 ~ (b_sz, y1_sz)
        np.dot(self.X, self.W1, out=self.Z1)  # Z1 = X @ W1   (preallocated out)
        # Y1 ~ (b_sz, y1_sz)
        self.Y1[:] = self._f1(self.Z1)  # Y1 = f1(Z1)   (generic activation)
        # Z2 ~ (b_sz, y2_sz)
        np.dot(self.Y1, self.W2, out=self.Z2)  # Z2 = Y1 @ W2  (preallocated out)
        # Y2 ~ (b_sz, y2_sz)
        self.Y2[:] = self._f2(self.Z2)  # Y2 = f2(Z2)   (generic activation)
        return self.Y2

    def backward(self, Y_true, lr=1e-3):
        # E2 = dL/dZ2 = dloss(Y2, Y_true) * f2'(Z2)   (generic, identical to Net)
        # E2 ~ (b_sz, y2_sz)
        self.E2[:] = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 = (E2 @ W2.T) * f1'(Z1)
        # E1 ~ (b_sz, y1_sz)
        np.dot(self.E2, self.W2.T, out=self.E1)  # E1 = E2 @ W2.T  (preallocated out)
        self.E1 *= self._df1(self.Z1)  # *= f1'(Z1)      (generic derivative)

        # weight gradients (preallocated out)
        # dL_dW2 ~ (y1_sz, y2_sz)
        np.dot(self.Y1.T, self.E2, out=self.dL_dW2)  # dL_dW2 = Y1.T @ E2
        # dL_dW1 ~ (x_sz, y1_sz)
        np.dot(self.X.T, self.E1, out=self.dL_dW1)  # dL_dW1 = X.T  @ E1

        # fused SGD step: scale grads in place by lr, then subtract (no temp alloc)
        self.dL_dW2 *= lr
        self.dL_dW1 *= lr
        self.W2 -= self.dL_dW2
        self.W1 -= self.dL_dW1


nn_fm = NetFixedMem(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
W2: float64 (16, 2)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)


In [54]:
# Closest possible adaptation of `Net` to MNIST digit classification.
#
# What CHANGES vs. Net (and why):
#   - sizes: x_sz=784 (28x28 pixels), y2_sz=10 (one score per digit class).
#   - _f2: linear -> softmax, so the 10 outputs form a probability distribution.
#   - _loss: mean squared error -> mean cross-entropy (the natural classification loss).
#   - weight init: scaled by 1/sqrt(fan_in) (He/Xavier) instead of plain randn,
#     otherwise 784 inputs make the pre-activations huge and softmax saturates.
#
# What STAYS IDENTICAL to Net (the elegant part):
#   - _f1/_df1 (ReLU), forward(), backward().
#   - _dloss returns (Y_predicted - Y_true) / N, and _df2 returns ones, EXACTLY as
#     in Net. For softmax + cross-entropy the output error dL/dZ2 collapses to
#     (softmax - one_hot)/N == (Y2 - Y_true)/N -- the same form as linear + MSE.
#     So we fold the softmax/CE derivative into this pair and reuse backward() verbatim.
#
# Y_true is expected one-hot: shape (batch, 10). Use one_hot(labels) to build it.
class DigitsRecognizerNet:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        # W1 ~ (x_sz, y1_sz)
        self.W1 = np.random.randn(x_sz, y1_sz) * np.sqrt(2.0 / x_sz)  # He init (ReLU)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")

        # W2 ~ (y1_sz, y2_sz)
        self.W2 = np.random.randn(y1_sz, y2_sz) * np.sqrt(
            1.0 / y1_sz
        )  # Xavier-ish (softmax)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # softmax (row-wise, numerically stable)
        # subtract the row-wise max for numerical stability
        # (fine bc doftmax is shift-invariant: subtracting any constant c from a row's logits gives the identical result )
        A = A - A.max(axis=1, keepdims=True)

        E = np.exp(A)

        return E / E.sum(axis=1, keepdims=True)

    @staticmethod
    def _df2(A):  # folded into _dloss for softmax+CE -> identity
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = self.X @ self.W1  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = self.Y1 @ self.W2  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # mean cross-entropy over batch
        eps = 1e-12
        return -(Y_true * np.log(Y_predicted + eps)).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):  # softmax+CE: dL/dZ2 == (Y2 - Y_true)/N
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1

        self.W2 += -lr * self.dL_dW2
        self.W1 += -lr * self.dL_dW1

    @staticmethod
    def one_hot(labels, n_classes=10):  # integer labels (batch,) -> (batch, n_classes)
        Y = np.zeros((labels.shape[0], n_classes))
        Y[np.arange(labels.shape[0]), labels] = 1.0
        return Y

    def predict(self, X):  # class index with highest probability
        return self.forward(X).argmax(axis=1)


dnn = DigitsRecognizerNet(x_sz=784, y1_sz=128, y2_sz=10)

W1: float64 (784, 128)
W2: float64 (128, 10)
